<a href="https://colab.research.google.com/github/iamtrask/abcGPT/blob/main/notebooks/train_dual_medium_sts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# abcGPT dual-source — medium (Shakespeare + TinyStories char)

alt_mixed Uniform on a friendlier second corpus. TinyStories is much simpler than Wikipedia at char-level (synthetic narrative, simple grammar), so the wiki-XL config is overkill. This sits between the compact 6M build and the 50M wiki-XL build:
- `n_layer=6, n_embd=512` (~25M total params, ~12.5M per slot)
- `max_iters=12000` (half the wiki-XL run)
- `dropout=0.05` (alt_mixed already regularizes via alpha-stochasticity)
- `learning_rate=8e-4`, `gradient_accumulation_steps=2` (effective batch 128)

Expected: val ~1.20-1.30 on the mixed corpus. Compute estimate: ~2 hours on T4.

**Just the training**, no analysis prelude. Hit Run All.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --quiet zstandard tiktoken

In [ ]:
import os
if not os.path.exists('/content/abcGPT'):
    !git clone --depth 1 https://github.com/iamtrask/abcGPT.git /content/abcGPT
else:
    !cd /content/abcGPT && git pull --rebase --autostash
%cd /content/abcGPT

In [ ]:
!python data/shakespeare_tinystories_char/prepare.py

## Run directory (tagged `-medium-sts` so this run is distinct from the others)

In [ ]:
RUN_ID = None   # set explicitly to resume, e.g. "20260519-XXXXXX-medium-sts"

In [ ]:
import time, os, glob
DRIVE_ROOT = '/content/drive/MyDrive/abcGPT/runs'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if 'RUN_ID' not in dir() or RUN_ID is None:
    RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-medium-sts'
elif not RUN_ID.endswith('-medium-sts'):
    RUN_ID = RUN_ID + '-medium-sts'
OUT_DIR_DUAL = f'{DRIVE_ROOT}/{RUN_ID}'
os.makedirs(OUT_DIR_DUAL, exist_ok=True)
print('run dir:', OUT_DIR_DUAL)
snaps = sorted(glob.glob(os.path.join(OUT_DIR_DUAL, '*.pt.zst')))
print(f'  {len(snaps)} existing snapshots' + (' (will resume)' if snaps else ' (fresh run)'))

## Train

On a T4 GPU expect roughly 2 hours for 12000 iters with the medium model. Pass time will be ~55-60s/pass.

In [ ]:
!python train_dual.py config/train_shakespeare_tinystories_dual_alt_mixed_medium.py \
    --out_dir=$OUT_DIR_DUAL \
    --batch_mode=alt_mixed \
    --first_pass_corpus=shake \
    --mix_distribution=uniform